# 🇪🇹 ENA News Web Scraper
### Web Scraping Project — Ethiopian News Agency (ena.et)

**Name:** Abraham Gebeyehu  
**Date:** 2026-06-19

---

## Project Brief

> Extract news articles from [ena.et](https://www.ena.et/web/eng/) and save the data to CSV.
> The deliverable must include titles, links, dates, and any other useful fields.

## Pipeline Overview

| Step | Section | Purpose |
|------|---------|--------|
| 1 | Imports | Scraping and data libraries |
| 2 | Configuration | All tunable settings in one place |
| 3 | Helper functions | HTTP, date parsing, word count |
| 4 | Scraping functions | Three-level crawl: homepage → category → article |
| 5 | Orchestration | Single entry point for the full crawl |
| 6 | Execute | Trigger the live crawl |
| 7 | Data cleaning | Type enforcement, missing-value audit |
| 8 | NumPy analytics | Descriptive stats, IQR outliers, histogram |
| 9 | Pandas analytics | Group-by summaries, keyword/category filtering |
| 10 | Export | Write CSV and XLSX deliverables |
| 11 | Conclusion | Results summary and techniques demonstrated |


## Step 1 — Imports


In [ ]:
import re
import sys
import time
from datetime import datetime
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np

print(f"requests {requests.__version__}  |  "
      f"pandas {pd.__version__}  |  numpy {np.__version__}")


## Step 2 — Configuration

All tunable settings are defined here so no magic values are buried inside functions.

- **`FETCH_FULL_CONTENT`** — when `True`, follows every article link for full body text.
  Set `False` for a fast structural test using only the listing-page preview.
- **`MAX_ARTICLES_PER_CATEGORY`** — set to a small integer for a quick test run;
  `None` means scrape everything.


In [ ]:
# --- Target site ---------------------------------------------------------
BASE_URL = "https://www.ena.et"
HOME_URL = "https://www.ena.et/web/eng/"

# URL slugs that identify a news category on the ENA site
VALID_SLUGS = {
    "politics", "social", "economy",
    "sport", "technology", "environment", "feature",
}

# Mimic a real browser to avoid 403 responses
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}

# --- Request behaviour ---------------------------------------------------
REQUEST_DELAY   = 0.5   # seconds between requests (politeness / rate-limit avoidance)
REQUEST_RETRIES = 2     # attempts before giving up on a URL

# --- Run options ---------------------------------------------------------
FETCH_FULL_CONTENT        = True   # follow each article link for full body text
MAX_ARTICLES_PER_CATEGORY = None   # None = no limit; set an int for a quick test

# --- Output paths --------------------------------------------------------
OUTPUT_CSV  = "ena_news_articles.csv"
OUTPUT_XLSX = "ena_news_articles.xlsx"

# Set True to print selector diagnostics for the first article fetched
DEBUG_CONTENT = False

print(f"Target : {HOME_URL}")
print(f"Cats   : {sorted(VALID_SLUGS)}")
print(f"Limit  : {MAX_ARTICLES_PER_CATEGORY or 'no limit'} articles/category")


## Step 3 — Helper Functions

| Function | Responsibility |
|---|---|
| `safe_get(url)` | HTTP GET with timeout, retries, and a polite delay |
| `extract_image_url(img_tag)` | Resolves lazy-loaded image URLs |
| `parse_date(raw_text)` | Extracts and parses a date from messy article text |
| `count_words(text)` | Word count, safe against `None`/empty input |


In [ ]:
# Extracts a date substring from surrounding text such as
# "Addis Ababa, June 18, 2026 (ENA) — ...". Three alternatives cover
# every date format observed on the live ENA site.
_DATE_RE = re.compile(
    r'(?:(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{1,2},?\s+\d{4})'
    r'|(?:\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{4})'
    r'|(?:\d{4}-\d{2}-\d{2})',
    re.IGNORECASE,
)

# strptime formats tried in order; first match wins
_DATE_FORMATS = (
    "%B %d, %Y",   # June 18, 2026
    "%b %d, %Y",   # Jun 18, 2026
    "%B %d %Y",    # June 18 2026
    "%b %d %Y",    # Jun 18 2026
    "%d %B %Y",    # 18 June 2026
    "%d %b %Y",    # 18 Jun 2026
    "%Y-%m-%d",    # 2026-06-18
)


def safe_get(url, retries=REQUEST_RETRIES):
    """Return a requests.Response for *url*, or None after all retries fail."""
    for attempt in range(retries + 1):
        try:
            time.sleep(REQUEST_DELAY)
            r = requests.get(url, headers=HEADERS, timeout=20)
            r.raise_for_status()
            return r
        except requests.RequestException as exc:
            tag = f"retry {attempt + 1}" if attempt < retries else "failed"
            print(f"  [{tag}] {url} -> {exc}")
    return None


def extract_image_url(img_tag):
    """Return the real image URL from an <img> tag, checking lazy-load attributes."""
    if not img_tag:
        return None
    for attr in ("src", "data-src", "data-original", "data-lazy-src"):
        val = img_tag.get(attr, "")
        if val and val.startswith(("http", "/")):
            return urljoin(BASE_URL, val)
    return None


def parse_date(raw_text):
    """
    Extract a date from *raw_text* and return a pandas Timestamp.

    ENA date fields contain extra surrounding text (city name, agency
    attribution, article ID). The _DATE_RE regex isolates the date
    substring before strptime is called. Without this step every row
    produces NaT because strptime cannot parse the full string.
    Returns pd.NaT if no recognisable date is found.
    """
    if not raw_text:
        return pd.NaT
    cleaned = re.sub(r"\s+", " ", str(raw_text)).strip()
    match = _DATE_RE.search(cleaned)
    if not match:
        return pd.NaT
    date_str = match.group(0).strip().rstrip(",")
    for fmt in _DATE_FORMATS:
        try:
            return pd.Timestamp(datetime.strptime(date_str, fmt))
        except ValueError:
            continue
    return pd.NaT


def count_words(text):
    """Return the word count of *text*, or 0 for None/empty input."""
    return len(text.split()) if text else 0


# Smoke-test parse_date() before touching the network
_SMOKE = [
    ("Addis Ababa, June 18, 2026 (ENA) — Ethiopia will...", "2026-06-18"),
    ("Jun 18, 2026 1402",                                    "2026-06-18"),
    ("18 June 2026",                                         "2026-06-18"),
    ("2026-06-18",                                           "2026-06-18"),
    ("",                                                      "NaT"),
]
print("parse_date() self-test:")
all_ok = True
for raw, expected in _SMOKE:
    result = parse_date(raw)
    ok = (str(result.date()) == expected) if result is not pd.NaT else (expected == "NaT")
    all_ok = all_ok and ok
    print(f"  {'OK' if ok else 'FAIL':4}  {raw!r:55} -> {result}")
print("All tests passed." if all_ok else "One or more tests FAILED.")


## Step 4 — Core Scraping Functions

The crawl follows three nested levels mirroring the site's structure:

```
ENA homepage
 └── Category page  (Politics, Sport, Economy, …)
      └── Paginated listing pages
           └── Individual article page  (title, date, body, image)
```

- **`get_categories()`** — discovers all category URLs from the homepage.
- **`get_category_pages()`** — collects every paginated listing URL for one category.
- **`get_full_article()`** — scrapes body text from a single article page.
- **`scrape_category()`** — ties the above together for one category.


In [ ]:
# A single Session carries cookies across all requests in one crawl run.
# Liferay sets session cookies on the homepage; article pages may require
# them to return 200 instead of a redirect or 403.
_SESSION = requests.Session()
_SESSION.headers.update(HEADERS)


def safe_get(url, retries=REQUEST_RETRIES):
    """Return a requests.Response for *url* using the shared session, or None on failure."""
    for attempt in range(retries + 1):
        try:
            time.sleep(REQUEST_DELAY)
            r = _SESSION.get(url, timeout=20)
            r.raise_for_status()
            return r
        except requests.RequestException as exc:
            tag = f"retry {attempt + 1}" if attempt < retries else "failed"
            print(f"  [{tag}] {url} -> {exc}")
    return None


def get_categories():
    """
    Visit the ENA homepage and return {label: url} for every news category.
    Visiting the homepage first seeds the session with Liferay cookies that
    article pages may require.
    """
    r = safe_get(HOME_URL)
    if not r:
        sys.exit("Cannot reach the ENA homepage. Check your network connection.")

    soup = BeautifulSoup(r.text, "html.parser")
    categories = {}
    for a in soup.find_all("a", href=True):
        slug = a["href"].rstrip("/").split("/")[-1].lower()
        if slug in VALID_SLUGS:
            label = a.get_text(strip=True) or slug.title()
            categories[label] = urljoin(BASE_URL, a["href"])

    if not categories:
        sys.exit("No categories found. The site structure may have changed.")

    print(f"Found {len(categories)} categories: {list(categories.keys())}")
    return categories


def get_category_pages(category_url):
    """
    Return every paginated listing-page URL for *category_url*.

    ENA runs on Liferay whose pagination links contain substrings such as
    'cur=', '_cur_', 'page=', or 'p_p_id'. We scan the first page,
    collect matching links into a set (automatic de-duplication), and
    always include the category URL itself as page 1.
    """
    r = safe_get(category_url)
    if not r:
        return [category_url]

    soup = BeautifulSoup(r.text, "html.parser")
    pages = {category_url}
    for a in soup.select("a[href]"):
        href = a["href"]
        if any(p in href for p in ("cur=", "_cur_", "page=", "p_p_id", "_com_liferay")):
            pages.add(urljoin(BASE_URL, href))

    return list(pages)


def get_full_article(article_url):
    """
    Open *article_url* and return its full body text, or None on failure.

    Selector chain is ordered from most specific to broadest fallback.
    '.portlet-body' is intentionally broad — it is the outermost Liferay
    content container and acts as the last resort before <main>.

    When DEBUG_CONTENT is True the function prints which selector matched
    and — if nothing matched — dumps all div class names from the page so
    the correct selector can be identified and added.
    """
    r = safe_get(article_url)
    if not r:
        return None

    soup = BeautifulSoup(r.text, "html.parser")

    _SELECTORS = [
        ".journal-content-article",
        ".asset-content",
        ".asset-abstract",
        ".portlet-body article",
        ".article-content",
        "article",
        ".portlet-body",
        ".portlet-content",
        "main",
    ]
    content = next((soup.select_one(s) for s in _SELECTORS if soup.select_one(s)), None)

    if DEBUG_CONTENT and not hasattr(get_full_article, "_debugged"):
        get_full_article._debugged = True
        matched = next((s for s in _SELECTORS if soup.select_one(s)), "NONE MATCHED")
        print(f"\n[DEBUG] First article  : {article_url}")
        print(f"[DEBUG] Winning selector: {matched}")
        if content:
            print(f"[DEBUG] Content preview : {content.get_text(' ', strip=True)[:300]}")
        else:
            seen = {cls for div in soup.find_all('div', class_=True)
                    for cls in div['class'] if len(cls) > 4}
            print(f"[DEBUG] Div classes on page: {sorted(seen)}")

    return content.get_text(" ", strip=True) if content else None


def scrape_category(category_name, category_url,
                     fetch_full_content=True, max_articles=None):
    """
    Scrape every article from *category_url* across all pagination pages.

    Parameters
    ----------
    category_name     : str        Label stored in the 'category' column.
    category_url      : str        URL of the category's first listing page.
    fetch_full_content: bool       Follow each article link for full body text.
    max_articles      : int|None   Per-category cap; None means no limit.

    Returns a list of dicts, one per article.
    """
    articles = []
    pages = get_category_pages(category_url)
    print(f"  -> {len(pages)} listing page(s) found")

    for page_url in pages:
        if max_articles and len(articles) >= max_articles:
            break

        r = safe_get(page_url)
        if not r:
            continue

        soup = BeautifulSoup(r.text, "html.parser")
        tiles = soup.select("div.ena_display_item")

        for tile in tiles:
            if max_articles and len(articles) >= max_articles:
                break

            title_tag   = tile.select_one(".ena_display_item_title")
            date_tag    = tile.select_one(".ena_display_item_date")
            preview_tag = tile.select_one(".ena_display_item_content")
            link_tag    = tile.select_one("a[href]")
            img_tag     = tile.select_one("img")

            title     = title_tag.get_text(" ", strip=True)   if title_tag   else None
            raw_date  = date_tag.get_text(" ", strip=True)    if date_tag    else None
            preview   = preview_tag.get_text(" ", strip=True) if preview_tag else None
            link      = urljoin(BASE_URL, link_tag["href"])   if link_tag    else None
            image_url = extract_image_url(img_tag)

            full_content = (
                get_full_article(link)
                if fetch_full_content and link else None
            )

            # raw_date is used only locally for parsing; it is not stored in
            # the DataFrame since the cleaned 'date' column supersedes it.
            articles.append({
                "category"  : category_name,
                "title"     : title,
                "date"      : parse_date(raw_date),
                "preview"   : preview,
                "content"   : full_content,
                "word_count": count_words(full_content or preview),
                "has_image" : image_url is not None,
                "image_url" : image_url,
                "link"      : link,
                "scraped_at": pd.Timestamp.now(),
            })

    return articles


## Step 5 — Orchestration

`scrape_ena()` is the single public entry point: discover categories,
scrape all articles, de-duplicate cross-category duplicates, sort newest-first.


In [ ]:
def scrape_ena(fetch_full_content=True, max_articles_per_category=None):
    """Run the full ENA scrape and return a clean, sorted DataFrame."""
    categories = get_categories()
    all_rows = []

    for name, url in categories.items():
        print(f"\nScraping [{name}]  {url}")
        rows = scrape_category(
            name, url,
            fetch_full_content=fetch_full_content,
            max_articles=max_articles_per_category,
        )
        print(f"  collected {len(rows)} articles")
        all_rows.extend(rows)

    if not all_rows:
        print("No articles scraped. Check your network connection.")
        return pd.DataFrame()

    df = pd.DataFrame(all_rows)
    df = df.drop_duplicates(subset=["link"], keep="first")
    df = df.sort_values("date", ascending=False, na_position="last").reset_index(drop=True)
    return df


## Step 6 — Run the Scraper

A full run with `FETCH_FULL_CONTENT = True` will take several minutes
because of the per-request delay. Set `MAX_ARTICLES_PER_CATEGORY = 5`
in Step 2 for a quick structural test before committing to the full crawl.


In [ ]:
df = scrape_ena(
    fetch_full_content=FETCH_FULL_CONTENT,
    max_articles_per_category=MAX_ARTICLES_PER_CATEGORY,
)

if df.empty:
    sys.exit("Nothing scraped — check your network or the ENA site structure.")

print(f"\nScrape complete. Shape: {df.shape}")
df.head()


## Step 7 — Data Cleaning & Validation


In [ ]:
# Enforce expected column types; coerce silently rather than raise on edge cases
df["date"]       = pd.to_datetime(df["date"], errors="coerce")
df["word_count"] = df["word_count"].astype(int)
df["has_image"]  = df["has_image"].astype(bool)

# Truncated preview for quick visual inspection in the notebook
df["preview_short"] = df["preview"].fillna("").str.slice(0, 120).str.strip() + "..."

print("Missing values per column:")
print(df.isna().sum())
print(f"\nDuplicate links : {df['link'].duplicated().sum()} (should be 0)")
print(f"Date range      : {df['date'].min()} -> {df['date'].max()}")
print(f"Categories      : {df['category'].unique().tolist()}")


## Step 8 — NumPy Analytics

1. Descriptive statistics on article word counts.
2. IQR-based outlier detection (standard statistical method).
3. Short/Medium/Long bucketing with `np.select` and a histogram.


In [ ]:
word_counts = df["word_count"].to_numpy()

# --- 1. Descriptive statistics ------------------------------------------
p25, p75 = np.percentile(word_counts, [25, 75])
print("Word-count statistics:")
print(f"  mean   : {np.mean(word_counts):.1f}")
print(f"  median : {np.median(word_counts):.1f}")
print(f"  std    : {np.std(word_counts):.1f}")
print(f"  Q1/Q3  : {p25:.1f} / {p75:.1f}")

# --- 2. IQR outlier detection -------------------------------------------
# Standard rule: flag any value more than 1.5*IQR beyond Q1 or Q3
iqr = p75 - p25
is_outlier = (word_counts < p25 - 1.5 * iqr) | (word_counts > p75 + 1.5 * iqr)
df["is_length_outlier"] = is_outlier
print(f"\nLength outliers : {is_outlier.sum()} of {len(df)}")

# --- 3a. Short / Medium / Long bucketing --------------------------------
df["length_category"] = np.select(
    [word_counts < p25,
     (word_counts >= p25) & (word_counts <= p75),
     word_counts > p75],
    ["Short", "Medium", "Long"],
    default="Unknown",
)
print("\nArticles per length bucket:")
print(df["length_category"].value_counts())

# --- 3b. Word-count histogram -------------------------------------------
counts, edges = np.histogram(word_counts, bins=5)
print("\nWord-count distribution (5 bins):")
for n, lo, hi in zip(counts, edges[:-1], edges[1:]):
    print(f"  {lo:7.0f} - {hi:7.0f} words : {n} article(s)")


## Step 9 — Pandas Analytics


In [ ]:
def articles_per_category(df):
    """Article counts and key stats per category, sorted by volume."""
    summary = (
        df.groupby("category")
        .agg(
            article_count=("title",      "count"),
            avg_words    =("word_count", "mean"),
            with_image   =("has_image",  "sum"),
            latest_date  =("date",       "max"),
        )
        .sort_values("article_count", ascending=False)
        .reset_index()
    )
    summary["avg_words"] = summary["avg_words"].round(0).astype(int)
    return summary


def top_articles_by_length(df, n=10):
    """Return the *n* longest articles by word count."""
    return (
        df[["category", "title", "date", "word_count", "link"]]
        .sort_values("word_count", ascending=False)
        .head(n)
        .reset_index(drop=True)
    )


def filter_by_category(df, name):
    """Case-insensitive filter to a single category."""
    return df[df["category"].str.lower() == name.lower()].reset_index(drop=True)


def filter_by_keyword(df, keyword):
    """Case-insensitive keyword search across title and preview."""
    kw = keyword.lower()
    mask = (
        df["title"].str.lower().str.contains(kw, na=False)
        | df["preview"].str.lower().str.contains(kw, na=False)
    )
    return df[mask].reset_index(drop=True)


def articles_by_date(df):
    """Count articles published per calendar day, newest first."""
    return (
        df.dropna(subset=["date"])
        .groupby(df["date"].dt.date)
        .size()
        .rename("articles_published")
        .reset_index()
        .rename(columns={"date": "publish_date"})
        .sort_values("publish_date", ascending=False)
    )


print("Articles per category:")
display(articles_per_category(df))

print("\nTop 10 longest articles:")
display(top_articles_by_length(df, n=10))

print("\nPublishing activity (most recent 10 days):")
display(articles_by_date(df).head(10))


In [ ]:
# Keyword search example
results = filter_by_keyword(df, "Ethiopia")
print(f"Articles mentioning 'Ethiopia': {len(results)}")
display(results[["category", "title", "date"]].head(5))

# Category deep-dive example
economy_df = filter_by_category(df, "Economy")
print(f"\nEconomy articles: {len(economy_df)}")
display(economy_df[["title", "date", "word_count", "length_category"]].head(5))


## Step 10 — Export to CSV

The `content` column is excluded from the export to keep files manageable;
it remains in the `df` variable for in-notebook use.
`utf-8-sig` encoding ensures special characters display correctly in Excel.


In [ ]:
export_columns = [
    "category", "title", "link", "date",
    "preview", "content", "word_count", "length_category", "is_length_outlier",
    "has_image", "image_url", "scraped_at",
]
export_df = df[export_columns]

export_df.to_csv(OUTPUT_CSV,  index=False, encoding="utf-8-sig")
export_df.to_excel(OUTPUT_XLSX, index=False, engine="openpyxl")

print(f"Exported {len(export_df)} rows -> {OUTPUT_CSV}  |  {OUTPUT_XLSX}")
export_df.head()


## Step 11 — Summary & Conclusion


In [ ]:
print("=" * 60)
print("  ENA SCRAPE SUMMARY")
print("=" * 60)
print(f"  Total articles      : {len(df)}")
print(f"  Categories          : {df['category'].nunique()}")
print(f"  Date range          : {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"  Articles with image : {int(df['has_image'].sum())}")
print(f"  Avg word count      : {df['word_count'].mean():.0f} words")
print(f"  Length outliers     : {int(df['is_length_outlier'].sum())}")
print(f"  Output              : {OUTPUT_CSV}  |  {OUTPUT_XLSX}")
print("=" * 60)


### Techniques demonstrated

- **Multi-level crawling** — homepage → category listing → individual article (three link levels).
- **Robust date extraction** — `_DATE_RE` isolates the date substring from surrounding city names,
  agency attributions, and article IDs before `strptime` is called. This is the root fix for
  the NaT values produced by the original version.
- **Complete pagination** — `get_category_pages()` detects all Liferay pager links so no
  per-category article cap is imposed by missing pages.
- **Article body scraping** — a selector fallback chain handles the multiple Liferay layout
  variants ENA uses, ensuring the `content` column is populated rather than left blank.
- **Defensive HTTP** — retries, timeouts, and `None`-safe field access prevent a single
  bad page from terminating the whole run.
- **NumPy statistics** — IQR outlier detection and histogram binning done directly on the
  underlying array.
- **Clean CSV/XLSX deliverable** — types enforced, duplicates removed, UTF-8-sig encoding
  for Excel compatibility.
